In [1]:
import random
from rdkit import Chem
from molpher.core import MolpherMol, MolpherAtom
from molpher.core.morphing.operators import MorphingOperator
from rdkit.Chem.EnumerateStereoisomers import EnumerateStereoisomers, StereoEnumerationOptions
from rdkit.Chem import rdChemReactions
from rdkit.Chem import rdmolops
from rdkit.Chem import Descriptors  
from molpher.core import ExplorationTree as ETree

class OxidizeAldehydeToAcid(MorphingOperator):
    def __init__(self):
        super(OxidizeAldehydeToAcid, self).__init__()
        self._name = "Oxidize Aldehyde to Acid"
        self._target_carbons = [] 
        self.PATTERN = Chem.MolFromSmarts("[CX3H1](=O)[#6,#1]")

    def setOriginal(self, mol):
        super(OxidizeAldehydeToAcid, self).setOriginal(mol)
        self._target_carbons = []
        
        if not self.original:
            return
            
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: 
            return
            
        matches = rdkit_mol.GetSubstructMatches(self.PATTERN)
        for match in matches:
            self._target_carbons.append(match[0])

    def morph(self):
        if not self.original: 
            return None
            
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: 
            return None
        
        if not self._target_carbons:
            return MolpherMol(other=rdkit_mol)
            
        idx_c = random.choice(self._target_carbons)
        
        try:
            rw_mol = Chem.RWMol(rdkit_mol)
            
            new_o_idx = rw_mol.AddAtom(Chem.Atom(8))
            
            rw_mol.AddBond(idx_c, new_o_idx, Chem.BondType.SINGLE)
            
            new_mol = rw_mol.GetMol()
            
            for idx in [idx_c, new_o_idx]:
                atom = new_mol.GetAtomWithIdx(idx)
                atom.SetNoImplicit(False)
                atom.SetNumExplicitHs(0)
        
            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)
            
            return MolpherMol(other=new_mol)
            
        except Exception as e:
            return MolpherMol(other=rdkit_mol)
        
    def getName(self):
        return self._name

oxidize_aldehyde_op = OxidizeAldehydeToAcid()    

test_aldehyde_molecules = {
    "1. Προπανάλη (Καθαρή Αλδεΰδη)": "CCC=O",
    "2. Ακετόνη (Κετόνη - Πρέπει να αγνοηθεί)": "CC(C)=O",
    "3. Ν,Ν-διμεθυλοφορμαμίδιο (Αμίδιο - Πρέπει να αγνοηθεί)": "CN(C)C=O",
    "4. Μυρμηκικός μεθυλεστέρας (Εστέρας - Πρέπει να αγνοηθεί)": "COC=O"
}

print("=== STARTING OXIDIZE ALDEHYDE TESTING ===")
for name, smiles in test_aldehyde_molecules.items():
    test_mol = Chem.MolFromSmiles(smiles)
    if test_mol is None:
        continue
        
    mol = MolpherMol(smiles)
    oxidize_aldehyde_op.setOriginal(mol)
    product = oxidize_aldehyde_op.morph()
    
    print(f"\n{name}")
    print(f"  SOURCE: {mol.getSMILES()}")
    
    if product and product.getSMILES() != mol.getSMILES():
        print(f"  TARGET: {product.getSMILES()}")
    else:
        print("  TARGET: No change (Safe - Ignored)")
print("\n=========================================")

=== STARTING OXIDIZE ALDEHYDE TESTING ===

1. Προπανάλη (Καθαρή Αλδεΰδη)
  SOURCE: CCC=O
  TARGET: CCC(=O)O

2. Ακετόνη (Κετόνη - Πρέπει να αγνοηθεί)
  SOURCE: CC(C)=O
  TARGET: No change (Safe - Ignored)

3. Ν,Ν-διμεθυλοφορμαμίδιο (Αμίδιο - Πρέπει να αγνοηθεί)
  SOURCE: CN(C)C=O
  TARGET: No change (Safe - Ignored)

4. Μυρμηκικός μεθυλεστέρας (Εστέρας - Πρέπει να αγνοηθεί)
  SOURCE: COC=O
  TARGET: No change (Safe - Ignored)

